<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/10_MEMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Modelos de Markov de máxima entropía</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">De modelos generativos a discriminativos</span>](#de-modelos-generativos-a-discriminativos)
- [<span style="color: #005C99;">El modelo MEMM</span>](#el-modelo-memm)
- [<span style="color: #005C99;">Características</span>](#caracteristicas)
- [<span style="color: #005C99;">Entrenamiento</span>](#entrenamiento)
- [<span style="color: #005C99;">Decodificación: greedy y Viterbi</span>](#decodificacion-greedy-y-viterbi)
- [<span style="color: #005C99;">Evaluación</span>](#evaluacion)
- [<span style="color: #005C99;">¿Qué aprendió el modelo?</span>](#que-aprendio-el-modelo)
- [<span style="color: #005C99;">Limitaciones: el sesgo de etiqueta</span>](#limitaciones-el-sesgo-de-etiqueta)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import cess_esp

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

_ = nltk.download("cess_esp", quiet=True)

<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

El HMM del cuaderno 09 etiquetó CESS-ESP con un 95.7 % de exactitud, pero para manejar las palabras desconocidas tuvimos que recurrir a un truco: reemplazarlas por clases según su forma (`<UNK-mente>`, `<UNK-Mayús>`...). El problema de fondo es que el HMM explica cada palabra **solo** a partir de su etiqueta, $P(w_t \mid q_t)$, así que no puede combinar de forma natural pistas como:

- la palabra misma, su sufijo y su prefijo,
- si empieza con mayúscula o contiene dígitos,
- las palabras vecinas,
- la etiqueta anterior.

Los **modelos de Markov de máxima entropía** (MEMM; McCallum, Freitag & Pereira, 2000; Ratnaparkhi, 1996) resuelven esto combinando dos ideas que ya conocemos: el **clasificador de máxima entropía** (regresión logística multinomial, cuaderno 07) y la **decodificación de Viterbi** (cuaderno 09).

<a id="de-modelos-generativos-a-discriminativos"></a>
<h2 style="color: #007ACC;">De modelos generativos a discriminativos</h2>

| | HMM (generativo) | MEMM (discriminativo) |
|---|-------------------|------------------------|
| Qué modela | La probabilidad conjunta $P(O, Q)$ | Directamente $P(Q \mid O)$ |
| Factorización | $\prod_t P(q_t \mid q_{t-1})\, P(o_t \mid q_t)$ | $\prod_t P(q_t \mid q_{t-1}, O, t)$ |
| Características | Solo la identidad de la palabra | Cualquier característica de toda la oración |
| Entrenamiento | Contar | Optimizar una regresión logística |
| Análogo en clasificación | Naive Bayes | Regresión logística |

La relación HMM ↔ MEMM es la misma que vimos entre Naive Bayes y regresión logística en el módulo 1: el modelo generativo tiene que explicar cómo se producen las observaciones, mientras que el discriminativo solo aprende a **distinguir** las etiquetas a partir de ellas, y por eso puede usar características solapadas sin violar ningún supuesto de independencia.

<a id="el-modelo-memm"></a>
<h2 style="color: #007ACC;">El modelo MEMM</h2>

Un MEMM estima, para cada posición $t$, la distribución de la etiqueta dada la etiqueta anterior y las observaciones, con un clasificador de máxima entropía:

$$P(q_t = j \mid q_{t-1} = i, O, t) = \frac{\exp\left( \mathbf{w}_j \cdot \boldsymbol\phi(O, t, i) \right)}{\sum_{k} \exp\left( \mathbf{w}_k \cdot \boldsymbol\phi(O, t, i) \right)}$$

donde $\boldsymbol\phi(O, t, i)$ es un vector de características que puede mirar **toda la oración** $O$ alrededor de la posición $t$, además de la etiqueta anterior $i$.

La probabilidad de la secuencia completa es el producto de estas probabilidades locales:

$$P(Q \mid O) = \prod_{t=1}^{T} P(q_t \mid q_{t-1}, O, t)$$

```text
HMM:   q1 → q2 → q3            MEMM:   q1 → q2 → q3
       ↓    ↓    ↓                     ↑ ↗  ↑ ↗  ↑
       o1   o2   o3                    o1   o2   o3   (y cualquier otra parte de O)
```

En el HMM las flechas van de las etiquetas a las palabras (las etiquetas *generan* las palabras); en el MEMM van de las palabras a las etiquetas (las palabras *condicionan* las etiquetas).

In [3]:
# --- Carga de CESS-ESP con etiquetas gramaticales simplificadas -------------------------
# Las etiquetas originales siguen el estándar EAGLES (p. ej. "ncms000" = nombre común masculino
# singular). Las reducimos a categorías universales usando su primera letra.
MAPA_EAGLES = {"n": "NOUN", "v": "VERB", "a": "ADJ", "d": "DET", "s": "ADP", "r": "ADV", "p": "PRON",
               "c": "CONJ", "f": "PUNCT", "z": "NUM", "w": "NUM", "i": "INTJ"}


def etiqueta_universal(eagles):
    eagles = eagles.lower()
    if eagles.startswith("np"):
        return "PROPN"
    if eagles.startswith(("va", "vs")):  # haber y ser como auxiliares
        return "AUX"
    return MAPA_EAGLES.get(eagles[0], "X")


SIGNOS = {"-Fpa-": "(", "-Fpt-": ")", "-fe-": '"'}
# Conservamos las mayúsculas: son una pista valiosa para reconocer nombres propios
corpus = [[(SIGNOS.get(w, w), etiqueta_universal(t)) for w, t in s if not w.startswith("*")]
          for s in cess_esp.tagged_sents()]

rng = np.random.default_rng(SEED)
orden = rng.permutation(len(corpus))
corte = int(0.9 * len(corpus))
train = [corpus[i] for i in orden[:corte]]
test = [corpus[i] for i in orden[corte:]]

ETIQUETAS = sorted({t for s in train for _, t in s})
t2i = {t: i for i, t in enumerate(ETIQUETAS)}
print(f"Entrenamiento: {len(train)} oraciones | Prueba: {len(test)} oraciones | {len(ETIQUETAS)} etiquetas")

Entrenamiento: 5427 oraciones | Prueba: 603 oraciones | 14 etiquetas


<a id="caracteristicas"></a>
<h2 style="color: #007ACC;">Características</h2>

El extractor de características recibe la oración, la posición y la etiqueta anterior, y devuelve un diccionario, como en el cuaderno 07. Incluimos lo que el HMM no podía usar directamente:

In [4]:
def caracteristicas(palabras, i, etiqueta_previa):
    w = palabras[i]
    p = w.lower()
    f = {
        "palabra=" + p: 1,
        "suf2=" + p[-2:]: 1,
        "suf3=" + p[-3:]: 1,
        "suf4=" + p[-4:]: 1,
        "pre3=" + p[:3]: 1,
        "etiqueta_previa=" + etiqueta_previa: 1,
        "palabra_previa=" + (palabras[i - 1].lower() if i > 0 else "<S>"): 1,
        "palabra_siguiente=" + (palabras[i + 1].lower() if i + 1 < len(palabras) else "</S>"): 1,
    }
    if w[0].isupper():
        f["mayúscula_inicial"] = 1
        if i > 0:
            f["mayúscula_no_inicial"] = 1
    if any(c.isdigit() for c in w):
        f["tiene_dígito"] = 1
    if "_" in w:
        f["multipalabra"] = 1
    return f


oracion = ["El", "presidente", "anunció", "rápidamente", "la", "reestructuración", "."]
caracteristicas(oracion, 3, "VERB")

{'palabra=rápidamente': 1,
 'suf2=te': 1,
 'suf3=nte': 1,
 'suf4=ente': 1,
 'pre3=ráp': 1,
 'etiqueta_previa=VERB': 1,
 'palabra_previa=anunció': 1,
 'palabra_siguiente=la': 1}

<a id="entrenamiento"></a>
<h2 style="color: #007ACC;">Entrenamiento</h2>

Para entrenar, cada palabra del corpus es un ejemplo de clasificación cuyas características incluyen la **etiqueta anterior verdadera** (la del corpus). Esto se conoce como *teacher forcing*: durante el entrenamiento el modelo siempre ve la etiqueta previa correcta.

`DictVectorizer` convierte los diccionarios en una matriz dispersa y `LogisticRegression` de scikit-learn entrena el clasificador de máxima entropía (con regularización L2, a diferencia del `MaxentClassifier` de NLTK).

In [5]:
def ejemplos(oraciones, usar_etiqueta_previa=True):
    X, y = [], []
    for s in oraciones:
        palabras, tags = zip(*s)
        for i in range(len(palabras)):
            previa = (tags[i - 1] if i > 0 else "<S>") if usar_etiqueta_previa else "-"
            X.append(caracteristicas(palabras, i, previa))
            y.append(tags[i])
    return X, y


X_dict, y_train = ejemplos(train)
vectorizador = DictVectorizer()
X_train = vectorizador.fit_transform(X_dict)
print(f"Ejemplos de entrenamiento: {X_train.shape[0]:,} | Características distintas: {X_train.shape[1]:,}")

inicio = time.perf_counter()
maxent = LogisticRegression(C=1.0, max_iter=300).fit(X_train, y_train)
print(f"Entrenado en {time.perf_counter() - inicio:.1f} s")
CLASES = list(maxent.classes_)

Ejemplos de entrenamiento: 169,685 | Características distintas: 81,361


Entrenado en 14.2 s


<a id="decodificacion-greedy-y-viterbi"></a>
<h2 style="color: #007ACC;">Decodificación: greedy y Viterbi</h2>

En prueba no conocemos la etiqueta anterior: es lo que estamos prediciendo. Hay dos formas de decodificar:

**Greedy (voraz):** de izquierda a derecha, se elige en cada posición la etiqueta más probable y se usa como etiqueta anterior para la siguiente. Es rápido, pero un error temprano se propaga.

**Viterbi:** se considera **cada** etiqueta anterior posible. Para cada posición $t$ y cada etiqueta previa $i$ calculamos la distribución $P(q_t = j \mid q_{t-1} = i, O, t)$, lo que da una matriz $N \times N$ que juega el papel de $a_{ij}\, b_j(o_t)$ en el HMM:

$$\log v_t(j) = \max_i \left[ \log v_{t-1}(i) + \log P(q_t = j \mid q_{t-1} = i, O, t) \right]$$

El algoritmo es idéntico al del cuaderno 09; solo cambia de dónde salen los puntajes.

In [6]:
def etiquetar_greedy(palabras):
    tags, previa = [], "<S>"
    for i in range(len(palabras)):
        x = vectorizador.transform([caracteristicas(palabras, i, previa)])
        previa = CLASES[int(maxent.predict_proba(x)[0].argmax())]
        tags.append(previa)
    return tags


def etiquetar_viterbi(palabras):
    T, N = len(palabras), len(CLASES)
    # Una fila de características por cada (posición, etiqueta previa): las predecimos todas juntas
    filas = [caracteristicas(palabras, 0, "<S>")]
    filas += [caracteristicas(palabras, t, previa) for t in range(1, T) for previa in CLASES]
    log_p = np.log(maxent.predict_proba(vectorizador.transform(filas)) + 1e-12)

    log_v = log_p[0]                          # (N,)  etiqueta de la primera palabra
    bp = np.zeros((T, N), dtype=int)
    for t in range(1, T):
        M = log_p[1 + (t - 1) * N: 1 + t * N]  # (N previas, N siguientes)
        puntajes = log_v[:, None] + M
        bp[t] = puntajes.argmax(axis=0)
        log_v = puntajes.max(axis=0)
    camino = [int(log_v.argmax())]
    for t in range(T - 1, 0, -1):
        camino.append(int(bp[t, camino[-1]]))
    return [CLASES[i] for i in camino[::-1]]


for frase in ["La vieja cierra la puerta .", "Bajo el bajo bajo el puente .",
              "El presidente Zelenski anunció rápidamente la reestructuración ."]:
    palabras = frase.split()
    print(" ".join(f"{w}/{t}" for w, t in zip(palabras, etiquetar_viterbi(palabras))))

La/DET vieja/ADJ cierra/NOUN la/DET puerta/NOUN ./PUNCT
Bajo/ADP el/DET bajo/NOUN bajo/ADP el/DET puente/NOUN ./PUNCT
El/DET presidente/NOUN Zelenski/PROPN anunció/VERB rápidamente/ADV la/DET reestructuración/NOUN ./PUNCT


La oración con palabras nuevas se etiqueta perfectamente. Las oraciones ambiguas del cuaderno 02 siguen siendo difíciles: ahora el modelo identifica *bajo* como sustantivo en la segunda posición, pero etiqueta *cierra* como sustantivo. Ningún modelo de este módulo "entiende" la oración; solo combinan pistas locales.

<a id="evaluacion"></a>
<h2 style="color: #007ACC;">Evaluación</h2>

Evaluamos sobre el mismo conjunto de prueba del cuaderno 09 (misma semilla y división). Como referencia, entrenamos también un clasificador de máxima entropía **sin la etiqueta anterior**: clasifica cada palabra de forma independiente, usando solo la ventana de palabras. Así medimos cuánto aporta modelar la secuencia.

In [7]:
X_dict_sin, y_sin = ejemplos(train, usar_etiqueta_previa=False)
vec_sin = DictVectorizer()
maxent_sin = LogisticRegression(C=1.0, max_iter=300).fit(vec_sin.fit_transform(X_dict_sin), y_sin)


def etiquetar_independiente(palabras):
    X = vec_sin.transform([caracteristicas(palabras, i, "-") for i in range(len(palabras))])
    return list(maxent_sin.predict(X))


vocab_train = {w.lower() for s in train for w, _ in s}
etiquetadores = {
    "MaxEnt sin etiqueta previa": etiquetar_independiente,
    "MEMM greedy": etiquetar_greedy,
    "MEMM Viterbi": etiquetar_viterbi,
}

filas = []
for nombre, etiquetador in etiquetadores.items():
    inicio = time.perf_counter()
    reales, preds, conocida = [], [], []
    for s in test:
        palabras, tags = zip(*s)
        reales += tags
        preds += etiquetador(list(palabras))
        conocida += [w.lower() in vocab_train for w in palabras]
    reales, preds, conocida = map(np.array, (reales, preds, conocida))
    filas.append({"etiquetador": nombre, "global": (reales == preds).mean(),
                  "conocidas": (reales == preds)[conocida].mean(),
                  "desconocidas": (reales == preds)[~conocida].mean(),
                  "tiempo (s)": time.perf_counter() - inicio})

resultados = pd.DataFrame(filas).set_index("etiquetador")
resultados.round(4)

,global,conocidas,desconocidas,tiempo (s)
etiquetador,,,,
MaxEnt sin etiqueta previa,0.9739,0.9802,0.9032,0.1218
MEMM greedy,0.9753,0.9814,0.9064,1.7026
MEMM Viterbi,0.9765,0.9826,0.9083,0.7015


Para comparar, estos fueron los resultados del cuaderno 09 sobre el mismo conjunto de prueba:

| Etiquetador | Global | Conocidas | Desconocidas |
|-------------|--------|-----------|--------------|
| Línea base (etiqueta más frecuente) | 0.8996 | 0.9585 | 0.2365 |
| HMM + Viterbi (clases de desconocidas) | 0.9571 | 0.9694 | 0.8192 |

Observaciones:

- Los tres modelos discriminativos superan al HMM, sobre todo en **palabras desconocidas**: las características de forma (sufijos, mayúsculas, dígitos) y las palabras vecinas se combinan de forma natural, sin trucos.
- Incluso el clasificador **sin etiqueta anterior** es muy fuerte: la ventana de palabras ya contiene buena parte del contexto que el HMM obtenía de las transiciones.
- Añadir la etiqueta anterior aporta una mejora adicional, y Viterbi aporta algo más que la decodificación greedy. Las diferencias entre los tres son pequeñas (unas décimas de punto), pero con unos 19 000 tokens de prueba equivalen a decenas de palabras bien etiquetadas.
- Sobre el tiempo: Viterbi hace $N$ veces más predicciones por palabra que greedy, pero aquí resulta más rápido porque agrupa todas las predicciones de una oración en una sola llamada al modelo, mientras que nuestro greedy llama al modelo palabra por palabra. Con implementaciones igual de optimizadas, greedy sería el más rápido.

<a id="que-aprendio-el-modelo"></a>
<h2 style="color: #007ACC;">¿Qué aprendió el modelo?</h2>

Como el MEMM es una regresión logística, podemos inspeccionar sus pesos. Para cada etiqueta, ¿qué características la favorecen más?

In [8]:
nombres = np.array(vectorizador.get_feature_names_out())
pd.DataFrame({
    etiqueta: nombres[np.argsort(-maxent.coef_[CLASES.index(etiqueta)])[:8]]
    for etiqueta in ["ADV", "VERB", "PROPN", "ADJ", "AUX", "NUM"]
})

,ADV,VERB,PROPN,ADJ,AUX,NUM
0,palabra=antes,suf2=ar,mayúscula_no_inicial,suf3=ble,pre3=hab,tiene_dígito
1,pre3=más,suf2=ir,mayúscula_inicial,pre3=bue,pre3=ser,palabra_siguiente=millones
2,suf4=ente,suf4=rles,etiqueta_previa=DET,suf2=al,pre3=hub,pre3=mié
3,multipalabra,suf2=ió,palabra_previa=(,suf3=oso,pre3=fue,palabra_siguiente=pasado
4,palabra=menos,suf4=rlos,palabra_previa=del,palabra=fuerte,pre3=hay,pre3=pes
5,pre3=tam,suf4=rnos,tiene_dígito,pre3=últ,pre3=era,palabra=dólares
6,palabra=entonces,suf2=an,multipalabra,etiqueta_previa=NOUN,pre3=sea,pre3=mar
7,pre3=tan,etiqueta_previa=AUX,palabra_siguiente=y,suf3=imo,suf3=es,pre3=eur


Aparecen los sufijos que esperaríamos (*-mente* para adverbios mediante `suf4=ente`, terminaciones verbales para verbos), la mayúscula a mitad de oración para nombres propios, los dígitos para numerales, y también características de la **etiqueta anterior** (por ejemplo, después de `AUX` es probable un verbo). El modelo aprendió de los datos las mismas reglas que diseñamos a mano para las clases `<UNK-...>` del HMM, y muchas más.

<a id="limitaciones-el-sesgo-de-etiqueta"></a>
<h2 style="color: #007ACC;">Limitaciones: el sesgo de etiqueta</h2>

Los MEMM tienen un problema conocido, el **sesgo de etiqueta** (*label bias*, Lafferty et al., 2001). Como cada distribución local $P(q_t \mid q_{t-1}, O, t)$ está **normalizada por separado**, toda la masa de probabilidad que llega a un estado debe repartirse entre sus sucesores, sin importar cuán bien encajen con la observación. Si desde un estado casi siempre se pasa al mismo siguiente estado (por ejemplo, después de cierta etiqueta casi siempre viene la misma), el modelo puede **ignorar la observación actual** porque la decisión ya está prácticamente tomada por la etiqueta anterior.

La solución son los **campos aleatorios condicionales** (CRF, *Conditional Random Fields*): usan las mismas características que un MEMM, pero normalizan **una sola vez sobre toda la secuencia**, en lugar de en cada paso:

$$P(Q \mid O) = \frac{1}{Z(O)} \exp\left( \sum_{t=1}^{T} \mathbf{w} \cdot \boldsymbol\phi(q_{t-1}, q_t, O, t) \right)$$

Los CRF lineales fueron el estándar en etiquetado de secuencias (POS, NER) durante la década de 2000, y siguen usándose como capa final de modelos neuronales (BiLSTM-CRF). La librería `sklearn-crfsuite` permite entrenarlos con diccionarios de características como los de este cuaderno.

**Resumen del módulo 2:**

| Modelo | Idea | Cuaderno |
|--------|------|----------|
| N-gramas | Probabilidad de la siguiente palabra con historia corta | 06 |
| Naive Bayes / MaxEnt en NLTK | Clasificación con características diseñadas a mano | 07 |
| HMM + forward | Modelo generativo de secuencias de etiquetas y palabras | 08 |
| Viterbi | Secuencia de etiquetas más probable con programación dinámica | 09 |
| MEMM | Clasificador discriminativo + Viterbi | 10 |

En el módulo 3 dejaremos de diseñar características a mano: las redes neuronales aprenderán representaciones de las palabras (*embeddings*) y del contexto (RNN, LSTM) directamente de los datos.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. [nltk.org/book](https://www.nltk.org/book/)
- Ratnaparkhi, A. (1996). A Maximum Entropy Model for Part-Of-Speech Tagging. *EMNLP*.
- McCallum, A., Freitag, D. & Pereira, F. (2000). Maximum Entropy Markov Models for Information Extraction and Segmentation. *ICML*.
- Lafferty, J., McCallum, A. & Pereira, F. (2001). Conditional Random Fields: Probabilistic Models for Segmenting and Labeling Sequence Data. *ICML*.